# Step 6: Locate TCR clonotypes spatially

This step performs clonotype-aware spatial analysis

## Setup and imports

In [ ]:
# Imports
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import squidpy as sq
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
import glob
import os
import shutil
import matplotlib.cm as cm
import scipy.stats
from scipy.stats import wilcoxon, zscore, gmean, pearsonr, norm, mannwhitneyu
from scipy import sparse
from statsmodels.stats.multitest import multipletests, fdrcorrection
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed
from matplotlib.patches import Patch

In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/Users/yyj/Doc/1_dod_dec25/processed_data/' #indir = '/path/to/xenium/raw_data'
outdir = '/Users/yyj/Doc/1_dod_dec25/out/' #outdir = '/path/to/integrated/processed_data'
figdir = '/Users/yyj/Doc/1_dod_dec25/figures/'

In [ ]:
# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

## Load data

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

## Define clones

Clones are chosen based on the following criteria:
- The clonotype chain (⍺β vs. γδ) matches the gene expression (CD4/CD8A vs. TRGC2)
- The clonotype passed the TCR probe quality dual criteria
- The clonotype generated statistically significant DE genes

In [ ]:
# all clones that passed the above criteria
pairs_map = {
    'γ Clone 15': ('clone15_', ['PBBCHF_PT'], 'γδ'),
    'γ Clone 17': ('clone17_', ['PBADJC_DX', 'PAZNRG_DX', 'PAZNRG_PT'], 'γδ'),
    'γ Clone 21': ('clone21_', ['PBBCHF_DX'], 'γδ'),
    'γ Clone 22': ('clone22_', ['PBBCHF_DX', 'PBBCHF_PT'], 'γδ'),
    'δ Clone 28': ('clone28_', ['PBBCHF_PT'], 'γδ'),
    'γ Clone 2': ('clone2_', ['PBBCHF_DX', 'PBBCHF_PT', 'PBBKFP_DX'], 'γδ'),
    'δ Clone 33': ('clone33_', ['PBBCHF_DX', 'PBBCHF_PT'], 'γδ'),
    'δ Clone 35': ('clone35_', ['PBBCHF_PT'], 'γδ'),
    'γ Clone 4': ('clone4_', ['PBBCHF_PT'], 'γδ'),
    'α Clone 39': ('clone39_', ['PBBCHF_PT'], '⍺β CD4'),
    'β Clone 84': ('clone84_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'α Clone 43': ('clone43_', ['PBBCHF_PT'], '⍺β CD8'),
    'α Clone 55': ('clone55_', ['PBBKFP_DX'], '⍺β CD8'),
    'β Clone 65': ('clone65_', ['PBBCHF_PT'], '⍺β CD8'),
    'α Clone 7': ('clone7_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'β Clone 82': ('clone82_', ['PBBKFP_PT'], '⍺β CD8'),
    'αβ Pair 1': ('clone45_', 'clone64_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
}

In [ ]:
# all clones that passed the above criteria
pairs_map = {
    'αβ Pair 1': ('clone45_', 'clone64_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 2': ('clone52_', 'clone60_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 3': ('clone51_', 'clone62_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
    'αβ Pair 4': ('clone57_', 'clone87_', ['PBBCHF_DX', 'PBBCHF_PT'], '⍺β CD8'),
}

## Visualize individual clones

In [ ]:
# Select individual clone for plotting
clone_name = "γ Clone 17" # clone of interest
value = pairs_map[clone_name]

if len(value) == 3:
    clone_targets = [value[0]]
    target_samples = value[1]
    tcr_type = value[2]
else:
    clone_targets = [value[0], value[1]]
    target_samples = value[2]
    tcr_type = value[3]

gene_target = "CD3E" # T cell marker of interest

In [ ]:
# Figure settings
n = len(target_samples)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
axes = np.atleast_1d(axes).flatten()

In [ ]:
# Helper function
def get_expr_sum(ad, prefix):
    genes = [g for g in ad.var_names if g.startswith(prefix)]
    if not genes:
        return np.zeros(ad.n_obs)
    X = ad[:, genes].X
    return X.sum(axis=1).A1 if scipy.sparse.issparse(X) else X.sum(axis=1)

In [ ]:
# Loop through samples and plot
for i, sample in enumerate(target_samples):
    ax = axes[i]
    
    sub = adata[adata.obs['sample'] == sample].copy()
    if sub.n_obs == 0:
        ax.axis('off'); continue

    # Target cell mask
    is_target = sub.obs['celltype'] == "T"

    # Clone condition (1 or 2)
    if len(clone_targets) == 1:
        c1 = get_expr_sum(sub, clone_targets[0]) > 0
        clone_mask = c1
        clone_label = clone_targets[0]
    elif len(clone_targets) == 2:
        c1 = get_expr_sum(sub, clone_targets[0]) > 0
        c2 = get_expr_sum(sub, clone_targets[1]) > 0
        clone_mask = c1 & c2
        clone_label = f"{clone_targets[0]} & {clone_targets[1]}"
    else:
        raise ValueError("clone_targets must have length 1 or 2.")

    # Optional gene condition (0 or 1)
    if gene_target is None:
        gene_mask = np.ones(sub.n_obs, dtype=bool)
        gene_label = None
    else:
        gene_mask = get_expr_sum(sub, gene_target) > 0
        gene_label = gene_target

    # Final double-positive definition
    is_dp = clone_mask & gene_mask & is_target

    # Coordinates
    x = sub.obsm['spatial'][:, 0]
    y = sub.obsm['spatial'][:, 1]

    # 1) Background (non-target)
    ax.scatter(x[~is_target], y[~is_target],
               c='#ffffb2', s=2, edgecolors='none', alpha=0.8)

    # 2) Target cells
    ax.scatter(x[is_target], y[is_target],
               c='#fed976', s=5, edgecolors='none', alpha=0.8)

    # 3) Double positives
    ax.scatter(x[is_dp], y[is_dp],
               c='#b10026', s=15, edgecolors='none', alpha=1)

    # Title
    if gene_label:
        title_txt = f"{sample}\n{clone_label} & {gene_label}+ (n={sum(is_dp)})"
    else:
        title_txt = f"{sample}\n{clone_label}+ (n={sum(is_dp)})"

    ax.set_title(title_txt, fontweight='bold')
    ax.axis('equal')
    ax.axis('off')

plt.tight_layout()

plt.savefig(f'{figdir}spatial_{clone_name}.png', 
    format='png', 
    transparent=True, 
    dpi=600,
    bbox_inches='tight')

plt.show()

## Run DE on individual clones

In [ ]:
# Helper function
def get_clone_sum(ad, prefix):
    genes = [g for g in ad.var_names if g.startswith(prefix)]
    if not genes:
        return np.zeros(ad.n_obs)
    X = ad[:, genes].X
    return X.sum(axis=1).A1 if scipy.sparse.issparse(X) else X.sum(axis=1)

In [ ]:
for sample_id in target_samples:
    print(f"\n{'='*25} Analyzing {sample_id} {'='*25}")
    
    sub_adata = adata[(adata.obs['sample'] == sample_id) & (adata.obs['celltype'] == "T")].copy()
    
    if sub_adata.n_obs == 0:
        print("No target cells found.")
        continue

    # Create DE object (exclude clone genes)
    non_clone_genes = [g for g in sub_adata.var_names if not g.startswith('clone')]
    adata_de = sub_adata[:, non_clone_genes].copy()
    
    # Build clone mask (1 or 2 clone prefixes)
    if len(clone_targets) == 1:
        c1 = get_clone_sum(sub_adata, clone_targets[0]) > 0
        clone_mask = c1
        clone_label = clone_targets[0]
    elif len(clone_targets) == 2:
        c1 = get_clone_sum(sub_adata, clone_targets[0]) > 0
        c2 = get_clone_sum(sub_adata, clone_targets[1]) > 0
        clone_mask = c1 & c2
        clone_label = f"{clone_targets[0]} & {clone_targets[1]}"
    else:
        raise ValueError("clone_targets must have length 1 or 2.")
    
    # Optional gene filter (0 or 1 gene)
    if gene_target is None:
        gene_mask = np.ones(sub_adata.n_obs, dtype=bool)
        gene_label = None
    else:
        gene_mask = get_clone_sum(sub_adata, gene_target) > 0
        gene_label = gene_target

    # Final double-positive definition
    is_dp = clone_mask & gene_mask
    n_dp = int(is_dp.sum())
    
    if n_dp < 3:
        label_txt = f"{clone_label}" if gene_label is None else f"{clone_label} & {gene_label}"
        print(f"\n--- {label_txt} ---")
        print(f"Skipping: Only {n_dp} double-positive cells found.")
        continue
        
    # Label Groups: 'DoublePos' vs 'Other'
    adata_de.obs['comparison_group'] = 'Other'
    adata_de.obs.loc[is_dp, 'comparison_group'] = 'DoublePos'
    
    # Run DE Test
    try:
        sc.tl.rank_genes_groups(
            adata_de, 
            groupby='comparison_group', 
            groups=['DoublePos'], 
            reference='Other', 
            method='wilcoxon'
        )
        
        top_genes = sc.get.rank_genes_groups_df(adata_de, group='DoublePos').head(5)
        
        label_txt = f"{clone_label}" if gene_label is None else f"{clone_label} & {gene_label}"
        print(f"\n--- {label_txt} (n={n_dp}) ---")
        for i, row in top_genes.iterrows():
            print(f"{i+1}. {row['names']} (p-adj: {row['pvals_adj']:.2e}, logFC: {row['logfoldchanges']:.2f})")
                
    except Exception as e:
        print(f"Error analyzing {clone_label}: {e}")

## Create dotplots for all clones
Anything beyong this points uses pairs_map rather than individual clones

In [ ]:
# Get genes to plot
genes_to_plot = [
    "CD4", "CD8A", "TRGC2",                             # lineage
    "SELL", "LEF1", "TCF7", "IL7R",                     # Naive / Tcm / TLS
    "THEMIS",                                           # TCR signal
    "MKI67",                                            # Proliferation
    "IFNG", "GZMH", "GZMB", "GZMK", "NKG7", "KLRD1",    # Cytotoxic
    "PDCD1", "LAG3", "TOX", "TIGIT"                     # Exhaustion
]

In [ ]:
# Build sample list from pairs_map
samples_in_pairs = []
for vals in pairs_map.values():
    # vals can be:
    # (clone_prefix, [samples], type) OR (clone1, clone2, [samples], type)
    if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        sample_list = vals[2]
    elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        sample_list = vals[1]
    else:
        continue

    for s in sample_list:
        if s not in samples_in_pairs:
            samples_in_pairs.append(s)

In [ ]:
# Loop through each sample and plot
for sample_id in samples_in_pairs:
    print(f"\nProcessing {sample_id}...")

    # Subset to sample + T cells
    sub = adata[(adata.obs["sample"] == sample_id) & (adata.obs["celltype"] == "T")].copy()

    if sub.n_obs == 0:
        print(f"Skipping {sample_id}: No T cells found.")
        continue

    # Initialize grouping
    sub.obs["clone_group"] = "Other T cells"

    # Assign clone groups
    for label, vals in pairs_map.items():
        # vals can be:
        # (clone1, clone2, [samples], type) OR (clone_prefix, [samples], type)
        if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
            clone_targets = [vals[0], vals[1]]
            allowed_samples = vals[2]
        elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
            clone_targets = [vals[0]]
            allowed_samples = vals[1]
        else:
            continue

        if sample_id not in allowed_samples:
            continue

        if len(clone_targets) == 1:
            clone_mask = get_clone_sum(sub, clone_targets[0]) > 0
        else:
            c1 = get_clone_sum(sub, clone_targets[0]) > 0
            c2 = get_clone_sum(sub, clone_targets[1]) > 0
            clone_mask = c1 & c2

        sub.obs.loc[clone_mask, "clone_group"] = label

    # Warn if no clone groups found
    if sub.obs["clone_group"].value_counts().drop("Other T cells", errors="ignore").sum() == 0:
        print(f"Warning: No specified clone groups found in {sample_id}")

    # Keep order of clones from pairs_map
    present = sub.obs["clone_group"].value_counts().index.tolist()
    order = [k for k in pairs_map.keys() if k in present] + (["Other T cells"] if "Other T cells" in present else [])

    # Add n/count to labels
    counts = sub.obs["clone_group"].value_counts().to_dict()
    label_map = {
        grp: (grp if grp == "Other T cells" else f"{grp} (n={counts.get(grp, 0)})")
        for grp in order
    }

    sub.obs["clone_group_label"] = sub.obs["clone_group"].map(label_map)
    ordered_labels = [label_map[g] for g in order]
    sub.obs["clone_group_label"] = pd.Categorical(
        sub.obs["clone_group_label"],
        categories=ordered_labels,
        ordered=True
    )

    # Validate genes exist
    current_genes = [g for g in genes_to_plot if g in sub.var_names]
    if len(current_genes) == 0:
        print(f"Skipping {sample_id}: none of genes_to_plot present.")
        continue

    dp = sc.pl.dotplot(
        sub,
        current_genes,
        groupby="clone_group_label",
        categories_order=ordered_labels,
        cmap="YlOrRd",
        return_fig=True,
        show=False,
        standard_scale="var",
    )

    dp = dp.style(
        smallest_dot=0.01,
        largest_dot=70,
    )

    # Access axes and style borders/ticks
    ax_dict = dp.get_axes()

    for _, ax in ax_dict.items():
        if ax is None:
            continue
        ax.tick_params(axis="both", which="both", width=0.5, length=2)

    main_ax = ax_dict.get("mainplot_ax", None)
    if main_ax is not None:
        for spine in main_ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    cax = ax_dict.get("color_legend_ax", None)
    if cax is not None:
        for spine in cax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color("#000000")

    fig = dp.fig
    fig.set_size_inches(9/2.54, (len(pairs_map) + 1.5)/2.54, forward=True)
    fig.tight_layout()
    fig.canvas.draw()

    display(fig)

    fig.savefig(
        f"{figdir}dotplot_{sample_id}_supp.pdf",
        bbox_inches="tight",
        transparent=True
    )
    plt.close(fig)

## Evaluate 4 pairs again neoTCR gene lists

In [ ]:
# Define signatures, pairs, and samples
signatures = {
    'Caushi': ['HAVCR2', 'ITGAE', 'ENTPD1', 'PDCD1', 'CTLA4', 'TOX2', 'ZNF683', 'GNLY', 'BATF', 'CXCL13'],
    'Oliveira': ['KRT86', 'RDH10', 'TYMS', 'HMOX1', 'GNG4', 'CXCL13', 'AFAP1L2', 'ACP5', 'MYO1E', 'LAYN', 'TNS3', 'TNFSF4', 'AKAP5', 'HAVCR2', 'ENTPD1', 'SLC2A8', 'ZBED2', 'MCM5', 'CAV1', 'GOLIM4', 'VCAM1', 'PON2', 'MTSS1', 'CD38', 'MS4A6A', 'TOX2', 'CSF1', 'GALNT2', 'FXYD2', 'PLPP1', 'LMCD1', 'MYL6B', 'LAG3', 'HLA-DRA', 'IGFLR1', 'CCDC50', 'CD27', 'KIAA1324', 'CDKN2A', 'CD70', 'ABHD6', 'CTLA4', 'PDCD1', 'GEM', 'NUSAP1', 'TOX', 'CXCR6', 'NMB', 'HOPX', 'CLIC3', 'INPP5F', 'SNAP47', 'TSHZ2', 'HLA-DMA', 'SIT1', 'HLA-DRB1', 'TUBB', 'PYCARD', 'ADGRG1', 'HLA-DQA1', 'PRF1', 'HLA-DPA1', 'PTMS', 'CKS1B', 'HIPK2', 'CHST12', 'LSP1', 'FAM3C', 'SLC1A4', 'NUDT1', 'DNPH1'],
    'Lowery': ['ATP10D', 'GZMB', 'ENTPD1', 'KIR2DL4', 'LAYN', 'HTRA1', 'CD70', 'CXCR6', 'HMOX1', 'ADGRG1', 'LRRN3', 'ACP5', 'CTSW', 'GALNT2', 'CARS', 'LAG3', 'TOX', 'PTPRCAP', 'ASB2', 'ITGB7', 'PTMS', 'CD8A', 'GPR68', 'NSMCE1', 'ABI3', 'SLC1A4', 'PLEKHF1', 'CD8B', 'CCL4', 'NKG7', 'CLIC3', 'NDFIP2', 'PLPP1', 'PCED1B', 'CXCL13', 'PDCD1', 'PRF1', 'HLA-DMA'],
    'Hanada': ['ENTPD1', 'CXCL13', 'HMOX1', 'PDCD1', 'LAYN', 'CD27', 'HAVCR2', 'TNFRSF9', 'MIR155HG', 'BATF', 'TIGIT', 'GZMH', 'CD70', 'TMEM121', 'LRRN3', 'NHS', 'TTN', 'ASB2', 'SIRPG', 'ANKS1B']
}

pairs = [
    ('clone45_', 'clone64_'),
    ('clone52_', 'clone60_'),
    ('clone51_', 'clone62_'),
    ('clone57_', 'clone87_')
]

samples = ['PBBCHF_DX', 'PBBCHF_PT']

pair_color = "#FFA63B"
other_color = "#E0E0E0"

In [ ]:
# Calculate signature scores
for name, genes in signatures.items():
    valid_genes = [g for g in genes if g in adata.var_names]
    if valid_genes:
        sc.tl.score_genes(adata, gene_list=valid_genes, score_name=f'score_{name}')
    else:
        print(f"Warning: No valid genes found for signature {name}")

In [ ]:
# Helper to identify cells
def get_clone_mask(ad, c1, c2):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    g2 = [g for g in ad.var_names if g.startswith(c2)]
    if not g1 or not g2:
        return np.zeros(ad.n_obs, dtype=bool)

    e1 = ad[:, g1].X.sum(axis=1)
    e2 = ad[:, g2].X.sum(axis=1)
    if hasattr(e1, 'A1'): e1 = e1.A1
    if hasattr(e2, 'A1'): e2 = e2.A1

    return (e1 > 0) & (e2 > 0)

In [ ]:
# Collect all tests for FDR
results_data = []
for sample in samples:
    sub = adata[(adata.obs['sample'] == sample) & (adata.obs['celltype'] == 'T')].copy()

    for (c1, c2) in pairs:
        is_pair = get_clone_mask(sub, c1, c2)
        n_pair = int(is_pair.sum())
        n_other = int(len(is_pair) - n_pair)

        if n_pair < 3 or n_other < 3:
            for sig_name in signatures.keys():
                results_data.append({
                    "sample": sample,
                    "pair": f"{c1}+{c2}",
                    "signature": sig_name,
                    "p": np.nan
                })
            continue

        for sig_name in signatures.keys():
            score_col = f"score_{sig_name}"
            s_pair = sub.obs.loc[is_pair, score_col]
            s_other = sub.obs.loc[~is_pair, score_col]

            stat, p = mannwhitneyu(s_pair, s_other, alternative='two-sided')
            results_data.append({
                "sample": sample,
                "pair": f"{c1}+{c2}",
                "signature": sig_name,
                "p": float(p)
            })

results_df = pd.DataFrame(results_data)

In [ ]:
# FDR across all valid p-values
valid = results_df["p"].notna()
reject, q = fdrcorrection(results_df.loc[valid, "p"].values, alpha=0.05)
results_df.loc[valid, "q"] = q
results_df.loc[valid, "significant_fdr"] = reject

def q_to_stars(qv):
    if pd.isna(qv):
        return ""
    if qv < 1e-3:
        return "***"
    if qv < 1e-2:
        return "**"
    if qv < 5e-2:
        return "*"
    return ""

results_df["stars"] = results_df["q"].apply(q_to_stars)

In [ ]:
# pair labels
pair_ids = [f"Pair{i+1}" for i in range(len(pairs))]
pair_label_map = {pair_ids[i]: f"{pairs[i][0]}+{pairs[i][1]}" for i in range(len(pairs))}
pair_to_id = {v: k for k, v in pair_label_map.items()}

In [ ]:
# Helper: sample -> timepoint
def sample_to_timepoint(s):
    s = str(s)
    if s.endswith("_DX"):
        return "DX"
    if s.endswith("_PT"):
        return "PT"
    return np.nan

In [ ]:
# Build long plot_df for boxplots
rows = []
for sample in samples:
    sub = adata[(adata.obs["sample"] == sample) & (adata.obs["celltype"] == "T")].copy()
    if sub.n_obs == 0:
        continue

    tp = sample_to_timepoint(sample)

    for (c1, c2) in pairs:
        pair_label = f"{c1}+{c2}"
        pid = pair_to_id[pair_label]

        is_pair = get_clone_mask(sub, c1, c2)
        if is_pair.sum() < 3 or (~is_pair).sum() < 3:
            continue

        for sig in signatures.keys():
            score_col = f"score_{sig}"
            if score_col not in sub.obs.columns:
                continue

            # Pair cells
            for v in sub.obs.loc[is_pair, score_col].values:
                rows.append({
                    "sample": sample,
                    "timepoint": tp,
                    "pair": pair_label,
                    "pair_id": pid,
                    "signature": sig,
                    "group": "Pair",
                    "score": float(v),
                })

            # Other cells
            for v in sub.obs.loc[~is_pair, score_col].values:
                rows.append({
                    "sample": sample,
                    "timepoint": tp,
                    "pair": pair_label,
                    "pair_id": pid,
                    "signature": sig,
                    "group": "Other",
                    "score": float(v),
                })

plot_df = pd.DataFrame(rows)

# Build tests_df expected by (1-83)
tests_df = results_df.copy()
tests_df["timepoint"] = tests_df["sample"].apply(sample_to_timepoint)
tests_df["pair_id"] = tests_df["pair"].map(pair_to_id)

In [ ]:
# Save test and plot data
tests_df.to_csv(f"{outdir}fig7f_neotcr_test_data.csv", index=True)
plot_df.to_csv(f"{outdir}fig7f_neotcr_plot_data.csv", index=True)

In [ ]:
# Plot 2x4 grid: rows=timepoint, cols=gene list, x=Pair1..Pair4
FONT_SIZE = 6

sig_order = list(signatures.keys())
tp_present = plot_df["timepoint"].dropna().unique().tolist()
tp_order = [t for t in ["DX", "PT"] if t in tp_present] + [t for t in tp_present if t not in ["DX", "PT"]]

fig, axes = plt.subplots(
    nrows=len(tp_order),
    ncols=len(sig_order),
    figsize=(12/2.54, 6/2.54),
    squeeze=False
)

for r, tp in enumerate(tp_order):
    for c, sig in enumerate(sig_order):
        ax = axes[r, c]
        d = plot_df[(plot_df["timepoint"] == tp) & (plot_df["signature"] == sig)].copy()

        if d.empty:
            ax.axis("off")
            continue

        sns.boxplot(
            data=d,
            x="pair_id", y="score", hue="group",
            order=pair_ids,
            hue_order=["Pair", "Other"],
            palette={"Pair": pair_color, "Other": other_color},
            showfliers=False,
            ax=ax,
            linewidth=0.5,  # box/whisker/cap/median line width
            boxprops={"linewidth": 0.5},
            whiskerprops={"linewidth": 0.5},
            capprops={"linewidth": 0.5},
            medianprops={"linewidth": 0.5},
        )

        # FDR stars for each pair in this panel
        y_min, y_max = ax.get_ylim()
        y_text = y_max + 0.1

        for i_pid, pid in enumerate(pair_ids):
            row = tests_df[
                (tests_df["timepoint"] == tp) &
                (tests_df["signature"] == sig) &
                (tests_df["pair_id"] == pid)
            ]
            if len(row) == 1:
                star = row["stars"].iloc[0]
                if star:
                    ax.text(
                        i_pid, y_text, star,
                        ha="center", va="top",
                        fontsize=FONT_SIZE, fontweight="bold"
                    )

        ax.set_title(f"{sig}", fontsize=FONT_SIZE)
        ax.set_xlabel("")
        ax.set_xticklabels(pair_ids, rotation=0, fontsize=FONT_SIZE)
        if c == 0:
            ax.set_ylabel("Score", fontsize=FONT_SIZE)
        else:
            ax.set_ylabel("")

        ax.tick_params(axis="both", which="both", width=0.5, length=2, labelsize=FONT_SIZE)

        ax.grid(False)
        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(False)

        # remove per-axis legend
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

# Optional pair key (kept, now fontsize 15)
pair_key = " | ".join([f"{pid}: {pair_label_map[pid]}" for pid in pair_ids])
fig.suptitle(f"Pair key: {pair_key}", y=1.02, fontsize=FONT_SIZE)

plt.tight_layout()
plt.savefig(
    f"{figdir}neotcr_2x4_timepoint_by_signature_boxes_by_pair.pdf",
    format="pdf",
    transparent=False,
    bbox_inches="tight",
)
plt.show()

## All clones and pairs vs. domains

In [ ]:
# Helper
def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return "Immune-rich"
    elif n_str in neuroblast_rich_clusters:
        return "Neuroblast-rich"
    else:
        return "Other"

In [ ]:
immune_rich_clusters = ["0", "4", "8", "9"]
neuroblast_rich_clusters = ["1", "2", "6", "7", "10", "12", "14", "16", "17", "18"]

categories_order = ["Immune-rich", "Neuroblast-rich", "Other"]

genes_to_plot = [
    "CCR7", "SELL", "IL7R",
    "THEMIS", "IFNG", "NKG7", "PDCD1", "TOX",
]

In [ ]:
# collect pseudobulk long tables per pair
pair_entries = [(label, vals) for label, vals in pairs_map.items() if isinstance(vals, tuple) and len(vals) == 4]
if len(pair_entries) == 0:
    raise ValueError("No pair entries (len(vals)==4) found in pairs_map.")

pair_pb_long = {}  # label -> (pb_long, tcr_type)
for label, vals in pair_entries:
    c1, c2, target_samples, tcr_type = vals

    sub_all = adata[
        (adata.obs["sample"].isin(target_samples)) &
        (adata.obs["celltype"] == "T")
    ].copy()
    if sub_all.n_obs == 0:
        continue

    clone_mask = get_clone_mask(sub_all, c1, c2)
    sub_all = sub_all[clone_mask].copy()
    if sub_all.n_obs == 0:
        continue

    sub_all.obs["domain"] = sub_all.obs["neigh_kmeans"].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs["domain"].notna()].copy()

    present = sub_all.obs["domain"].value_counts()
    categories_present = [c for c in categories_order if c in present.index]
    if len(categories_present) < 2:
        continue

    X = sub_all[:, genes_to_plot].X
    if hasattr(X, "toarray"):
        X = X.toarray()

    expr_df = pd.DataFrame(X, columns=genes_to_plot, index=sub_all.obs_names)
    meta_df = sub_all.obs[["sample", "domain"]].copy()
    meta_df["domain"] = pd.Categorical(meta_df["domain"], categories=categories_present, ordered=True)

    pb = pd.concat([meta_df, expr_df], axis=1)
    pb = pb.groupby(["sample", "domain"], observed=True)[genes_to_plot].mean().reset_index()
    pb_long = pb.melt(id_vars=["sample", "domain"], var_name="gene", value_name="expr")

    pair_pb_long[label] = (pb_long, tcr_type, categories_present)

In [ ]:
# Save plot data
pb_long.to_csv(f"{outdir}fig7g_boxplot_domain_data.csv", index=True)

In [ ]:
# make one figure: rows = pairs, cols = genes
plot_labels = [label for label, _ in pair_entries]
n_rows = len(plot_labels)
n_cols = len(genes_to_plot)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(12.6/2.54, 9/2.54),
    squeeze=False
)

for r, label in enumerate(plot_labels):
    payload = pair_pb_long.get(label, None)
    if payload is None:
        # no data for this pair -> blank row
        for c in range(n_cols):
            axes[r, c].axis("off")
        continue

    pb_long, tcr_type, categories_present = payload

    for c, gene in enumerate(genes_to_plot):
        ax = axes[r, c]
        d_gene = pb_long[pb_long["gene"] == gene]
        if d_gene.empty:
            ax.axis("off")
            continue

        sns.boxplot(
            data=d_gene,
            x="domain", y="expr",
            order=categories_present,
            color="lightgray",
            showfliers=False,
            ax=ax,
            width=0.8,
            linewidth=0.5,
            boxprops={"linewidth": 0.5},
            whiskerprops={"linewidth": 0.5},
            capprops={"linewidth": 0.5},
            medianprops={"linewidth": 0.5},
        )

        ax.tick_params(
            axis="both",
            which="both",
            width=0.5,
            length=2,
            labelsize=FONT_SIZE
        )

        ax.spines["bottom"].set_visible(True)
        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_linewidth(0.5)
        ax.spines["left"].set_linewidth(0.5)

        # titles on top row only
        if r == 0:
            ax.set_title(gene, fontsize=FONT_SIZE)
        else:
            ax.set_title("")

        # x labels only on bottom row
        if r == n_rows - 1:
            ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
        else:
            ax.set_xticklabels([])
            ax.set_xlabel("")

        # y labels only first column (pair label)
        if c == 0:
            ax.set_ylabel(f"{label}\n({tcr_type})", fontsize=FONT_SIZE)
        else:
            ax.set_ylabel("")

        ax.grid(False)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)

plt.tight_layout()
fig.subplots_adjust(wspace=0.8, hspace=0.1)

plt.savefig(
    f"{figdir}pseudobulk_all_pairs_domains.pdf",
    bbox_inches="tight",
    transparent=True
)
plt.show()

## Extra code chunks for additional visualization
Not included in the manuscript

In [ ]:
# Consensus neighbor composition by clone type

gene_target = "CD3E"

def get_neighbor_composition(adata, cell_mask, label_col="celltype"):
    connectivities = adata.obsp["spatial_connectivities"]
    target_indices = np.where(cell_mask)[0]
    if len(target_indices) == 0:
        return None

    all_neighbors = []
    labels = adata.obs[label_col].values
    for idx in target_indices:
        neighbor_indices = connectivities[idx].indices
        all_neighbors.extend(labels[neighbor_indices])

    if not all_neighbors:
        return None
    return pd.Series(all_neighbors).value_counts(normalize=True)

# Build sample -> clone targets per type (merge CD4/CD8 into αβ)
sample_to_type_clones = {}  # sample -> {type: [clone targets]}
for label, vals in pairs_map.items():
    # Expected:
    # (clone_prefix, [samples], type) OR (clone1, clone2, [samples], type)
    if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        clone_targets = [vals[0], vals[1]]
        allowed_samples = vals[2]
        clone_type = vals[3]
    elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        clone_targets = [vals[0]]
        allowed_samples = vals[1]
        clone_type = vals[2]
    else:
        continue

    if clone_type in ["⍺β CD8", "⍺β CD4"]:
        clone_type = "αβ"

    for s in allowed_samples:
        sample_to_type_clones.setdefault(s, {}).setdefault(clone_type, []).extend(clone_targets)

all_samples = sorted(sample_to_type_clones.keys())

# Collect per-sample compositions
type_groups = ["γδ", "αβ"]
sample_comps_by_type = {t: [] for t in type_groups}
sample_comps_other = []

for sample in all_samples:
    print(f"\nProcessing {sample}...")

    adata_full = adata[adata.obs["sample"] == sample].copy()
    if adata_full.n_obs == 0:
        print(f"Skipping {sample}: no cells found.")
        continue

    # Build spatial graph once per sample
    sq.gr.spatial_neighbors(adata_full, coord_type="generic", spatial_key="spatial")

    # Identify target cell mask (T cells only)
    is_target = adata_full.obs["celltype"] == "T"

    # Optional gene filter
    if gene_target is None:
        gene_mask = np.ones(adata_full.n_obs, dtype=bool)
    else:
        gene_mask = get_clone_sum(adata_full, gene_target) > 0

    # Build masks per type
    masks_by_type = {}
    for t in type_groups:
        masks_by_type[t] = np.zeros(adata_full.n_obs, dtype=bool)
        for ct in sample_to_type_clones.get(sample, {}).get(t, []):
            masks_by_type[t] |= (get_clone_sum(adata_full, ct) > 0)
        masks_by_type[t] = masks_by_type[t] & gene_mask & is_target

    # Other T cells = target cells not in any type mask
    any_clone = np.zeros(adata_full.n_obs, dtype=bool)
    for t in type_groups:
        any_clone |= masks_by_type[t]
    is_other_t = is_target & ~any_clone

    # Get neighbor compositions per type
    for t in type_groups:
        comp = get_neighbor_composition(adata_full, masks_by_type[t], label_col="celltype")
        if comp is not None:
            sample_comps_by_type[t].append(comp)

    comp_other = get_neighbor_composition(adata_full, is_other_t, label_col="celltype")
    if comp_other is not None:
        sample_comps_other.append(comp_other)

# Require at least one sample per group
if not any(sample_comps_by_type[t] for t in type_groups) or not sample_comps_other:
    print("No valid samples for consensus plot.")
else:
    # Equal-weighted mean across samples
    df = {}
    for t in type_groups:
        if sample_comps_by_type[t]:
            df_t = pd.concat(sample_comps_by_type[t], axis=1).fillna(0)
            df[t] = df_t.mean(axis=1)
        else:
            df[t] = pd.Series(dtype=float)

    df_other = pd.concat(sample_comps_other, axis=1).fillna(0)
    df["Other T cells"] = df_other.mean(axis=1)

    df = pd.DataFrame(df).fillna(0)

    # Top 10 overall + Other
    top_types = df.mean(axis=1).nlargest(10).index.tolist()
    df_top = df.loc[top_types]
    other_row = 1.0 - df_top.sum()
    df_top.loc["Other"] = other_row

    # Colors from celltype palette
    celltype_categories = list(adata.obs["celltype"].cat.categories)
    celltype_colors = list(adata.uns["celltype_colors"])
    color_dict = dict(zip(celltype_categories, celltype_colors))
    color_dict["Other"] = "#E6E6E6"

    # Plot (single figure)
    df_plot = df_top.T  # rows = γδ / αβ / Other T cells
    x = np.arange(len(df_plot.index))
    fig, ax = plt.subplots(figsize=(7, 7))

    for i, group in enumerate(df_plot.index):
        row = df_plot.loc[group]

        # Order neighbors by abundance within each group, keep "Other" on top
        ordered = row.sort_values(ascending=False).index.tolist()
        if "Other" in ordered:
            ordered.remove("Other")
            ordered.append("Other")

        bottom = 0.0
        for col in ordered:
            val = row[col]
            if val == 0:
                continue
            ax.bar(
                x[i], val, bottom=bottom,
                color=color_dict.get(col, "#E6E6E6"),
                edgecolor="white", linewidth=0.5,
                width=0.95
            )
            if val >= 0.05:
                ax.text(
                    x[i], bottom + val / 2, f"{val:.2f}",
                    ha="center", va="center", fontsize=15
                )
            bottom += val

    # X labels (fixed order)
    group_labels = ["γδ T", "αβ T", "Other T cells"]
    ax.set_xticks(x)
    ax.set_xticklabels(group_labels, rotation=0)
    ax.set_ylabel("Proportion of Neighbors")

    ax.set_frame_on(False)
    ax.grid(False)
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(False)

    legend_handles = [
        Patch(facecolor=color_dict.get(col, "#E6E6E6"), edgecolor="white", label=col)
        for col in df_plot.columns
    ]
    ax.legend(
        handles=legend_handles,
        title="Neighbor Cell Type",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(
        f"{figdir}top10_neighbor_clonotypes_celltype.png",
        format="png",
        transparent=True,
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
# Domain dotplot for the 4 pairs
immune_rich_clusters = ["0", "4", "8", "9"]
neuroblast_rich_clusters = ["1", "2", "6", "7", "10", "12", "14", "16", "17", "18"]

def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return "Immune-rich"
    elif n_str in neuroblast_rich_clusters:
        return "Neuroblast-rich"
    else:
        return "Other"

categories_order = ["Immune-rich", "Neuroblast-rich", "Other"]

# One dotplot per pair pooled across all its samples

genes_to_plot = [
    "CD4", "CD8A", "TRGC2",                    # lineage
    "SELL", "LEF1", "TCF7", "IL7R",           # Naive / Tcm / TLS
    "THEMIS",
    "MKI67",                                   # Proliferation
    "IFNG", "GZMH", "GZMB", "GZMK", "NKG7", "KLRD1",   # Cytotoxic
    "PDCD1", "LAG3", "TOX", "TIGIT"        # Exhaustion
]

for label, vals in pairs_map.items():
    if len(vals) != 4:  # skip clones, keep only pairs
        continue

    clone_targets = [vals[0], vals[1]]
    target_samples = vals[2]
    tcr_type = vals[3]

    sub_all = adata[(adata.obs["sample"].isin(target_samples)) & (adata.obs["celltype"] == "T")].copy()
    if sub_all.n_obs == 0:
        continue

    clone_mask = get_clone_mask(sub_all, *clone_targets)
    sub_all = sub_all[clone_mask].copy()
    if sub_all.n_obs == 0:
        continue

    sub_all.obs["domain"] = sub_all.obs["neigh_kmeans"].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs["domain"].notna()].copy()

    present = sub_all.obs["domain"].value_counts()
    if present.shape[0] < 2:
        continue

    # keep only categories that exist for this pair
    categories_present = [c for c in categories_order if c in present.index]
    if len(categories_present) < 2:
        continue

    genes = [g for g in genes_to_plot if g in sub_all.var_names]
    if len(genes) == 0:
        continue

    sc.pl.dotplot(
        sub_all,
        genes,
        groupby="domain",
        categories_order=categories_present,
        standard_scale="var",
        cmap="YlOrRd",
        title=f"{label} ({tcr_type}): 3-way domains",
        show=False,
    )
    plt.tight_layout()
    #plt.savefig(f"{figdir}dotplot_{label}_domains.png", dpi=600, bbox_inches="tight", transparent=True)
    plt.show()